# Extension Step 4 — DAGNN Task Graph Classifier

Trains a `DAGClassifier` (DAGNN, faithful to Thost & Chen ICLR 2021) on task graph realizations
using Leave-One-Out (LOO) cross-validation. For each recipe k, the model is trained on all other
recipes and tested on recipe k.

**Architecture**: text+visual node features → NodeFusionProjector → input projection →
DAGNNConv × L (attention + GRU, topological order) → MaxPool on target nodes → classifier.

**Prerequisites:**
- Step embeddings in `STEP_EMBEDDINGS_DIR` (output of Extension Step 1)
- Task graphs in `GRAPHS_DIR` (annotations submodule)
- EgoVLP checkpoint at `EGOVLP_CKPT`

**Output:**
- Pre-fusion cache in `CACHE_DIR` (built once, reused by all folds)
- Checkpoints in `STEP4_OUTPUT_DIR/checkpoints/fold_*_best.pt`
- Metrics in `STEP4_OUTPUT_DIR/results.csv`

In [ ]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Path constants (FIXED — do not modify) ─────────────────────────────────
DRIVE_ROOT           = '/content/drive/MyDrive/AML_Project'
REPO_DIR             = '/content/code'
EGOVLP_REPO          = '/content/EgoVLP'

EGOVLP_CKPT          = f'{DRIVE_ROOT}/models/egovlp.pth'
STEP_EMBEDDINGS_DIR  = f'{DRIVE_ROOT}/step1/step_embeddings'
ANNOTATIONS_PATH     = f'{REPO_DIR}/annotations/annotation_json/complete_step_annotations.json'
GRAPHS_DIR           = f'{REPO_DIR}/annotations/task_graphs'

CACHE_DIR            = f'{DRIVE_ROOT}/step4/cache'
STEP4_OUTPUT_DIR     = f'{DRIVE_ROOT}/step4/results'

# local copy of step embeddings (faster I/O than Drive during training)
LOCAL_EMBEDDINGS_DIR = '/content/step_embeddings'

print('Paths defined.')

In [ ]:
# ── 3. Clone repos (--recursive fetches annotations submodule) ────────────────
!git clone --recursive https://github.com/Laio95/aml-2025-mistake-detection.git {REPO_DIR}
!git clone https://github.com/showlab/EgoVLP.git {EGOVLP_REPO}

In [ ]:
# ── 4. Install dependencies ───────────────────────────────────────────────────
import torch, os
pt_version  = torch.__version__.split('+')[0]
cuda_str    = f"cu{torch.version.cuda.replace('.', '')}"
os.environ['TORCH'] = pt_version
os.environ['CUDA']  = cuda_str
print(f'PyTorch {pt_version}, CUDA {cuda_str}')

# torch-scatter / torch-sparse need precompiled binaries matching the runtime
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html -q
!pip install decord pytorchvideo fvcore iopath torch-geometric -q
!pip install -r {REPO_DIR}/requirements.txt -q
print('Dependencies installed.')

In [ ]:
# ── 5. Download ViT-B/16 checkpoint required by EgoVLP FrozenInTime ───────────
!mkdir -p {EGOVLP_REPO}/pretrained
!wget -q -nc -O {EGOVLP_REPO}/pretrained/jx_vit_base_p16_224-80ecf9dd.pth \
  https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-vitjx/jx_vit_base_p16_224-80ecf9dd.pth
!ls -lh {EGOVLP_REPO}/pretrained/

In [ ]:
# ── 6. Copy step embeddings to local storage (faster I/O than Drive) ──────────
import os
os.makedirs(LOCAL_EMBEDDINGS_DIR, exist_ok=True)
!cp -r {STEP_EMBEDDINGS_DIR}/* {LOCAL_EMBEDDINGS_DIR}/
print(f'Copied: {len(os.listdir(LOCAL_EMBEDDINGS_DIR))} files → {LOCAL_EMBEDDINGS_DIR}')

In [ ]:
# ── 7. (OPTIONAL) Clear pre-fusion cache ──────────────────────────────────────
# Run this cell ONLY if dataset_dag.py was modified since the last run.
# The cache stores Data(text_feats, vis_feats, matched_mask, edge_index, y) per recording.
# It is automatically rebuilt by prebuild_cache() at training start if missing.
#
# import shutil
# if os.path.exists(CACHE_DIR):
#     shutil.rmtree(CACHE_DIR)
#     os.makedirs(CACHE_DIR)
#     print(f'Cache cleared: {CACHE_DIR}')
print('(Cache clear skipped — uncomment above to enable)')

In [ ]:
# ── 8. WandB login ────────────────────────────────────────────────────────────
!wandb login

## Smoke test — 3 epochs (sanity check)

Verifica che:
- La pre-fusion cache venga costruita correttamente (384 file `.pt`)
- Il `DAGClassifier` (DAGNN) giri senza errori su tutti i 24 fold
- Loss e AUC abbiano valori sensati dopo 3 epoch

**Tempo stimato:** ~2-4 min totali.

In [ ]:
SMOKE_OUTPUT_DIR = f'{DRIVE_ROOT}/step4/smoke_test'

%%bash -s "$REPO_DIR" "$ANNOTATIONS_PATH" "$LOCAL_EMBEDDINGS_DIR" "$GRAPHS_DIR" "$EGOVLP_REPO" "$EGOVLP_CKPT" "$CACHE_DIR" "$SMOKE_OUTPUT_DIR"
cd $1

python -m extension.step4.train_dag_classifier \
    --annotations_path    "$2" \
    --step_embeddings_dir "$3" \
    --graphs_dir          "$4" \
    --egovlp_repo         "$5" \
    --egovlp_ckpt         "$6" \
    --cache_dir           "$7" \
    --output_dir          "$8" \
    --num_epochs   3  \
    --hidden_dim   128 \
    --num_layers   2  \
    --dropout      0.5 \
    --lr           1e-3 \
    --weight_decay 1e-4 \
    --batch_size   4  \
    --threshold    0.5 \
    --seed         42  \
    --num_workers  2

## Full LOO training — 50 epochs

Allena il `DAGClassifier` su tutti i 24 fold LOO (uno per ricetta).  
La pre-fusion cache è già stata costruita durante lo smoke test e viene riutilizzata.  
Il loop supporta il **resume**: se un checkpoint esiste già, il fold viene saltato.

**Tempo stimato:** ~5-8 min per fold × 24 fold ≈ 2-3 ore su T4.

In [ ]:
%%bash -s "$REPO_DIR" "$ANNOTATIONS_PATH" "$LOCAL_EMBEDDINGS_DIR" "$GRAPHS_DIR" "$EGOVLP_REPO" "$EGOVLP_CKPT" "$CACHE_DIR" "$STEP4_OUTPUT_DIR"
cd $1

python -m extension.step4.train_dag_classifier \
    --annotations_path    "$2" \
    --step_embeddings_dir "$3" \
    --graphs_dir          "$4" \
    --egovlp_repo         "$5" \
    --egovlp_ckpt         "$6" \
    --cache_dir           "$7" \
    --output_dir          "$8" \
    --num_epochs   50  \
    --hidden_dim   128 \
    --num_layers   2   \
    --dropout      0.5 \
    --lr           1e-3 \
    --weight_decay 1e-4 \
    --batch_size   4   \
    --threshold    0.5 \
    --seed         42  \
    --num_workers  2   \
    --enable_wandb

## Risultati — verifica e summary

In [ ]:
# ── Verifica checkpoint salvati ───────────────────────────────────────────────
import pathlib
ckpt_dir = pathlib.Path(STEP4_OUTPUT_DIR) / 'checkpoints'
ckpts = sorted(ckpt_dir.glob('*.pt'))
print(f'{len(ckpts)} checkpoint(s) salvati:')
for p in ckpts:
    print(f'  {p.name}')

In [ ]:
# ── Leggi e stampa results.csv ────────────────────────────────────────────────
import pandas as pd

csv_path = f'{STEP4_OUTPUT_DIR}/results.csv'
df = pd.read_csv(csv_path)
print(df.to_string(index=False))

In [ ]:
# ── Confronto con B2 baseline ─────────────────────────────────────────────────
import numpy as np

# B2 results (Transformer on step embeddings, no task graph structure)
b2 = {'AUC': (0.8092, 0.1118), 'F1': (0.7128, 0.1473), 'Accuracy': (0.6667, 0.1367)}

# B4 results (DAGNN on task graph realization)
numeric = df[df['fold'].apply(lambda x: str(x).isdigit())]
b4 = {
    'AUC':      (numeric['auc'].astype(float).mean(),      numeric['auc'].astype(float).std()),
    'F1':       (numeric['f1'].astype(float).mean(),       numeric['f1'].astype(float).std()),
    'Accuracy': (numeric['accuracy'].astype(float).mean(), numeric['accuracy'].astype(float).std()),
}

print('=== Tabella 5 — Task Verification ===')
print(f'{"Modello":<45} {"AUC":>15} {"F1":>15} {"Accuracy":>15}')
print('-' * 95)
print(f'{"Transformer (B2, no task graph)":<45} '
      f'{b2["AUC"][0]:.4f}±{b2["AUC"][1]:.4f}  '
      f'{b2["F1"][0]:.4f}±{b2["F1"][1]:.4f}  '
      f'{b2["Accuracy"][0]:.4f}±{b2["Accuracy"][1]:.4f}')
print(f'{"DAGNN (B4, task graph realization)":<45} '
      f'{b4["AUC"][0]:.4f}±{b4["AUC"][1]:.4f}  '
      f'{b4["F1"][0]:.4f}±{b4["F1"][1]:.4f}  '
      f'{b4["Accuracy"][0]:.4f}±{b4["Accuracy"][1]:.4f}')